**建立数据库**

In [ ]:
%%sql
-- v2 --

drop schema if exists payroll_management;
create schema payroll_management;
use payroll_management;

create table Employee
(
    EmployeeId     char(8)        not null comment '员工编号',
    EmployeeName   varchar(20)    not null comment '姓名',
    Department     varchar(30)    not null comment '部门',
    Position       varchar(30)    not null comment '职位',
    BasicSalary    decimal(10, 2) not null comment '基本工资',
    JoinDate       date           not null comment '入职日期',
    Contact        varchar(50) comment '联系方式',
    PaymentAccount varchar(50) comment '工资支付账户',
    primary key (EmployeeId)
) comment '员工';

create table Event
(
    EventId     char(4)     not null comment '事件类型编号',
    EventName   varchar(20) not null unique comment '事件类型',
    EventType   bool        not null comment '事件类别，0表示缺勤，1表示加班',
    Description varchar(200) comment '描述',
    primary key (EventId)
) comment '工作事件类型';

create table Absence_Record
(
    AbsenceId     char(10)     not null comment '记录编号',
    EmployeeId    char(8)      not null comment '员工编号',
    AbsenceType   char(4)      not null comment '缺勤类型编号',
    StartDateTime datetime     not null comment '开始时间',
    EndDateTime   datetime     not null comment '结束时间',
    Duration      int unsigned not null comment '缺勤时长（分钟）',
    Attachment    mediumtext comment '证明材料',
    primary key (AbsenceId),
    foreign key (EmployeeId) references Employee (EmployeeId),
    foreign key (AbsenceType) references Event (EventId)
) comment '缺勤记录';

create table Overtime_Record
(
    OvertimeId    char(10)     not null comment '记录编号',
    EmployeeId    char(8)      not null comment '员工编号',
    OvertimeType  char(4)      not null comment '加班类型编号',
    StartDateTime datetime     not null comment '加班开始时间',
    EndDateTime   datetime     not null comment '加班结束时间',
    Duration      int unsigned not null comment '加班时长（分钟）',
    Attachment    mediumtext comment '证明材料',
    primary key (OvertimeId),
    foreign key (EmployeeId) references Employee (EmployeeId),
    foreign key (OvertimeType) references Event (EventId)
) comment '加班记录';

create table Payroll_Item
(
    ItemId   char(6)     not null comment '项目编号',
    ItemName varchar(20) not null unique comment '项目名称',
    ItemType bool        not null comment '项目类型，0表示扣除项目，1表示奖励项目',
    ItemRule text        not null comment '计算规则描述',
    primary key (ItemId)
) comment '工资奖励/扣除项目';

create table Bonus_Record
(
    BonusId     char(14)      not null comment '记录编号',
    EmployeeId  char(8)       not null comment '员工编号',
    BonusType   char(6)       not null comment '奖金类型',
    BonusAmount decimal(8, 2) not null comment '金额',
    BonusDate   date comment '发放日期',
    IsSettled   bool          not null default 0 comment '是否已发放',
    primary key (BonusId),
    foreign key (EmployeeId) references Employee (EmployeeId),
    foreign key (BonusType) references Payroll_Item (ItemId)
) comment '奖金记录';

create table Deduction_Record
(
    DeductionId     char(14)      not null comment '扣款记录编号',
    EmployeeId      char(8)       not null comment '员工编号',
    DeductionType   char(6)       not null comment '扣款类型',
    DeductionAmount decimal(8, 2) not null comment '扣款金额',
    DeductionDate   date comment '扣款日期',
    IsSettled       bool          not null default 0 comment '是否已扣款',
    primary key (DeductionId),
    foreign key (EmployeeId) references Employee (EmployeeId),
    foreign key (DeductionType) references Payroll_Item (ItemId)
) comment '扣款记录';

create table Event_to_Payroll_Item
(
    MapId    int     not null auto_increment comment '映射编号',
    ItemId   char(6) not null comment '项目编号',
    EventId  char(4) not null comment '事件编号',
    IsActive bool    not null comment '是否启用',
    primary key (MapId),
    unique (ItemId, EventId),
    foreign key (ItemId) references Payroll_Item (ItemId),
    foreign key (EventId) references Event (EventId)
) comment '工资项目/工作事件映射';

create table Payment_Method
(
    PaymentMethodId   char(3)     not null comment '支付方式编号',
    PaymentMethodName varchar(20) not null unique comment '支付方式名称',
    Description       varchar(200) comment '描述',
    BankAccount       varchar(50) comment '银行账户信息',
    primary key (PaymentMethodId)
) comment '支付方式';

create table Payroll_Record
(
    PayrollId      char(14)       not null comment '工资单编号',
    EmployeeId     char(8)        not null comment '员工编号',
    PayrollDate    date           not null comment '发放日期',
    PaymentMethod  char(3)        not null comment '支付方式',
    BasicSalary    decimal(10, 2) not null comment '基本工资',
    TotalBonus     decimal(10, 2) not null comment '总奖金',
    TotalDeduction decimal(10, 2) not null comment '总扣款',
    NetSalary      decimal(10, 2) not null comment '实发工资',
    primary key (PayrollId),
    foreign key (EmployeeId) references Employee (EmployeeId),
    foreign key (PaymentMethod) references Payment_Method (PaymentMethodId)
) comment '工资单记录';


delimiter $$
create procedure checkDateTimeRange(
    in s datetime,
    in e datetime
)
begin
    if e <= s then
        signal sqlstate '45000'
            set message_text = 'EndDateTime must be > StartDateTime';
    end if;
end $$

create function calcDuration(
    s datetime,
    e datetime
)
    returns int
    deterministic
begin
    return timestampdiff(minute, s, e);
end $$

create trigger trg_Absence_type
    before insert
    on Absence_Record
    for each row
begin
    if (select EventType from Event where EventId = new.AbsenceType) = 1 then
        signal sqlstate '45000' set message_text = 'AbsenceType must refer to an absence event';
    end if;
end $$

create trigger trg_Overtime_type
    before insert
    on Overtime_Record
    for each row
begin
    if (select EventType from Event where EventId = new.OvertimeType) = 0 then
        signal sqlstate '45000' set message_text = 'OvertimeType must refer to an overtime event';
    end if;
end $$

create trigger trg_Absence_calc
    before insert
    on Absence_Record
    for each row
begin
    call checkDateTimeRange(new.StartDateTime, new.EndDateTime);
    set new.Duration = calcDuration(new.StartDateTime, new.EndDateTime);
end $$

create trigger trg_Overtime_calc
    before insert
    on Overtime_Record
    for each row
begin
    call checkDateTimeRange(new.StartDateTime, new.EndDateTime);
    set new.Duration = calcDuration(new.StartDateTime, new.EndDateTime);
end $$

create trigger trg_Bonus_type
    before insert
    on Bonus_Record
    for each row
begin
    if (select ItemType from Payroll_Item where ItemId = new.BonusType) = 0 then
        signal sqlstate '45000' set message_text = 'BonusType must refer to a reward item';
    end if;
end $$

create trigger trg_Deduction_type
    before insert
    on Deduction_Record
    for each row
begin
    if (select ItemType from Payroll_Item where ItemId = new.DeductionType) = 1 then
        signal sqlstate '45000' set message_text = 'DeductionType must refer to a deduction item';
    end if;
end $$

create trigger trg_Bonus_default_date
    before insert
    on Bonus_Record
    for each row
begin
    if new.BonusDate is null then
        set new.BonusDate = date_format(curdate(), '%Y-%m-28');
    end if;
end $$

create trigger trg_Deduction_default_date
    before insert
    on Deduction_Record
    for each row
begin
    if new.DeductionDate is null then
        set new.DeductionDate = date_format(curdate(), '%Y-%m-28');
    end if;
end $$

create trigger trg_Absence_type_update
    before update
    on Absence_Record
    for each row
begin
    if (select EventType from Event where EventId = new.AbsenceType) = 1 then
        signal sqlstate '45000' set message_text = 'AbsenceType must refer to an absence event';
    end if;
end $$

create trigger trg_Absence_calc_update
    before update
    on Absence_Record
    for each row
begin
    call checkDateTimeRange(new.StartDateTime, new.EndDateTime);
    set new.Duration = calcDuration(new.StartDateTime, new.EndDateTime);
end $$

create trigger trg_Overtime_type_update
    before update
    on Overtime_Record
    for each row
begin
    if (select EventType from Event where EventId = new.OvertimeType) = 0 then
        signal sqlstate '45000' set message_text = 'OvertimeType must refer to an overtime event';
    end if;
end $$

create trigger trg_Overtime_calc_update
    before update
    on Overtime_Record
    for each row
begin
    call checkDateTimeRange(new.StartDateTime, new.EndDateTime);
    set new.Duration = calcDuration(new.StartDateTime, new.EndDateTime);
end $$
delimiter ;

**生成mock数据**

1. `Employee`表
    1. 对`Employee`表生成随机记录
    2. 写入数据

In [ ]:
import csv
import pymysql
import random
import string
from datetime import date, timedelta
from pathlib import Path
from typing import Iterable


def ask_use_defaults() -> bool:
    choice = input('Use default names and jobs files? (Y/n): ').strip().lower()
    return choice not in {'n', 'not'}


def get_file_path(default_path: Path, label: str) -> Path:
    path_str = input(f'Path to {label} file (default {default_path.name}): ').strip()
    path = Path(path_str) if path_str else default_path
    if not path.is_file():
        raise FileNotFoundError(f'File not found: {path}')
    return path


def ask_insert_db() -> bool:
    choice = input('Insert generated records into MySQL? (y/N): ').strip().lower()
    return choice in {'y', 'yes'}


def get_db_config() -> dict:
    host = input('MySQL host (default localhost): ').strip() or 'localhost'
    port_str = input('MySQL port (default 3306): ').strip() or '3306'
    user = input('MySQL user: ').strip()
    password = input('MySQL password: ').strip()
    database = input('MySQL database name: ').strip()

    if not user or not password or not database:
        raise ValueError('MySQL user, password, and database are required.')

    try:
        port = int(port_str)
    except ValueError:
        raise ValueError('MySQL port must be an integer.')

    return {
        'host': host,
        'port': port,
        'user': user,
        'password': password,
        'database': database,
    }


def get_row_count(default: int = 64) -> int:
    rows = input(f'Amount of rows to read (default {default}): ').strip()
    return int(rows) if rows else default


def gen_random_indices(count: int, *, max_index: int) -> list[int]:
    if count > max_index:
        raise ValueError(f'Requested {count} rows but file has only {max_index} lines.')
    return random.sample(range(1, max_index + 1), count)


def read_lines(path: Path) -> list[str]:
    # Strip trailing newlines but preserve original order.
    with path.open(encoding='utf-8') as f:
        return [line.rstrip('\n') for line in f]


def read_jobs(path: Path) -> list[tuple[str, str]]:
    with path.open(encoding='utf-8', newline='') as f:
        reader = csv.reader(f)
        rows = [tuple(row) for row in reader if row]

    if rows and rows[0] and rows[0][0].lower() == 'dept':
        rows = rows[1:]

    jobs = [row for row in rows if len(row) >= 2]
    if not jobs:
        raise ValueError('jobs.csv is empty or invalid.')
    return jobs


def gen_unique_ids(count: int, length: int = 8) -> list[str]:
    alphabet = string.digits
    ids: set[str] = set()
    while len(ids) < count:
        ids.add(''.join(random.choices(alphabet, k=length)))
    return list(ids)


def gen_random_date(start: date, end: date) -> date:
    if end < start:
        raise ValueError('End date must not be before start date.')
    delta_days = (end - start).days
    return start + timedelta(days=random.randint(0, delta_days))


def gen_employee_records(names: list[str], jobs: list[tuple[str, str]]) -> list[dict]:
    join_start = date(2012, 1, 1)
    today = date.today()
    ids = gen_unique_ids(len(names))
    salaries = [float(f'{random.randrange(3000, 15001, 100):.2f}') for _ in names]

    records = []
    for i, name in enumerate(names):
        dept, position = random.choice(jobs)
        records.append(
            {
                'id': ids[i],
                'name': name,
                'dept': dept,
                'position': position,
                'salary': salaries[i],
                'join_date': gen_random_date(join_start, today).isoformat(),
            }
        )
    return records


def insert_into_mysql(
        records: Iterable[dict], *, host: str, port: int, user: str, password: str, database: str
) -> None:
    connection = pymysql.connect(
        host=host,
        port=port,
        user=user,
        password=password,
        database=database,
        charset='utf8mb4',
        autocommit=False,
    )
    try:
        with connection.cursor() as cursor:
            sql = (
                'INSERT INTO Employee (EmployeeId, EmployeeName, Department, Position, BasicSalary, JoinDate) '
                'VALUES (%s, %s, %s, %s, %s, %s)'
            )
            payload = [
                (
                    r['id'],
                    r['name'],
                    r['dept'],
                    r['position'],
                    r['salary'],
                    r['join_date'],
                )
                for r in records
            ]
            cursor.executemany(sql, payload)
        connection.commit()
    finally:
        connection.close()


default_names_path = Path(__file__).with_name('static') / 'names.txt'
default_jobs_path = Path(__file__).with_name('static') / 'jobs.csv'

use_defaults = ask_use_defaults()
names_path = default_names_path if use_defaults else get_file_path(default_names_path, 'names')
jobs_path = default_jobs_path if use_defaults else get_file_path(default_jobs_path, 'jobs')

jobs = read_jobs(jobs_path)
lines = read_lines(names_path)
count = get_row_count()
random_indices = gen_random_indices(count, max_index=len(lines))
selected_lines = [lines[i - 1] for i in random_indices]

employees = gen_employee_records(selected_lines, jobs)

print('Employee records:', employees)

if employees and ask_insert_db():
    cfg = get_db_config()
    insert_into_mysql(employees, **cfg)
    print('Inserted records into MySQL successfully.')

2. 建立基本字典表
    1. 对`Event`表写入基本记录
    2. 对`Payroll_Item`表写入基本记录
    3. 对`Payment_Method`表写入基本记录

In [ ]:
%%sql
-- Event --
insert into payroll_management.Event (EventId, EventName, EventType, Description) values ('AB01', '事假', 0, null);
insert into payroll_management.Event (EventId, EventName, EventType, Description) values ('AB02', '病假', 0, null);
insert into payroll_management.Event (EventId, EventName, EventType, Description) values ('BR01', '产假', 0, null);
insert into payroll_management.Event (EventId, EventName, EventType, Description) values ('BR02', '陪产假', 0, null);
insert into payroll_management.Event (EventId, EventName, EventType, Description) values ('FM01', '婚假', 0, null);
insert into payroll_management.Event (EventId, EventName, EventType, Description) values ('FM02', '丧假', 0, null);
insert into payroll_management.Event (EventId, EventName, EventType, Description) values ('LT01', '年假', 0, null);
insert into payroll_management.Event (EventId, EventName, EventType, Description) values ('LT02', '调休', 0, null);
insert into payroll_management.Event (EventId, EventName, EventType, Description) values ('OH01', '节假日加班', 1, null);
insert into payroll_management.Event (EventId, EventName, EventType, Description) values ('OR01', '休息日加班', 1, null);
insert into payroll_management.Event (EventId, EventName, EventType, Description) values ('OR02', '节假日轮转加班', 1, null);
insert into payroll_management.Event (EventId, EventName, EventType, Description) values ('OW01', '工作日加班', 1, null);
insert into payroll_management.Event (EventId, EventName, EventType, Description) values ('OW02', '工作日延时加班', 1, null);

In [ ]:
%%sql
-- Payroll_Item --
insert into payroll_management.Payroll_Item (ItemId, ItemName, ItemType, ItemRule) values ('ABS001', '短事假', 0, '按请假时长扣除当日基本工资的相应比例');
insert into payroll_management.Payroll_Item (ItemId, ItemName, ItemType, ItemRule) values ('ABS002', '长事假', 0, '按公司长事假制度执行 按请假天数扣除当日基本工资的100%');
insert into payroll_management.Payroll_Item (ItemId, ItemName, ItemType, ItemRule) values ('BRL001', '生育假', 0, '生育假期间按国家及公司产假制度发放 按基本工资的80%计发');
insert into payroll_management.Payroll_Item (ItemId, ItemName, ItemType, ItemRule) values ('FEL001', '家庭假', 0, '家庭事件假按公司制度全额计发 不扣薪');
insert into payroll_management.Payroll_Item (ItemId, ItemName, ItemType, ItemRule) values ('HOT001', '常规节假日加班', 1, '节假日加班按基本时薪的3倍支付');
insert into payroll_management.Payroll_Item (ItemId, ItemName, ItemType, ItemRule) values ('HOT002', '节假日加班', 1, '节假日加班按基本时薪的3倍支付 不得以调休代替');
insert into payroll_management.Payroll_Item (ItemId, ItemName, ItemType, ItemRule) values ('ILL001', '小病假', 0, '小病假按带薪比例发放 按基本工资的70%计发 不足部分扣除');
insert into payroll_management.Payroll_Item (ItemId, ItemName, ItemType, ItemRule) values ('ILL002', '大病假', 0, '大病假按长期病假制度执行 按基本工资的50%计发 不足部分扣除');
insert into payroll_management.Payroll_Item (ItemId, ItemName, ItemType, ItemRule) values ('LHT001', '调休', 0, '调休按工时抵扣 不产生工资扣减 当调休余额不足时按日薪扣除差额');
insert into payroll_management.Payroll_Item (ItemId, ItemName, ItemType, ItemRule) values ('LHT002', '年假', 0, '年假按工龄及公司制度正常计发 超额使用部分按日薪扣除');
insert into payroll_management.Payroll_Item (ItemId, ItemName, ItemType, ItemRule) values ('ROT001', '常规休息日加班', 1, '休息日加班按基本时薪的2倍支付 如安排调休则不另行支付');
insert into payroll_management.Payroll_Item (ItemId, ItemName, ItemType, ItemRule) values ('ROT002', '休息日加班', 1, '休息日加班按基本时薪的2倍支付 如调休抵扣则按调休优先');
insert into payroll_management.Payroll_Item (ItemId, ItemName, ItemType, ItemRule) values ('WOT001', '常规工作日加班', 1, '工作日加班按基本时薪的1.5倍支付');
insert into payroll_management.Payroll_Item (ItemId, ItemName, ItemType, ItemRule) values ('WOT002', '工作日加班', 1, '延时工作日加班按基本时薪的1.25倍支付');

In [ ]:
%%sql
-- Event_to_Payroll_Item --
insert into payroll_management.Event_to_Payroll_Item (MapId, ItemId, EventId, IsActive) values (1, 'ABS001', 'AB01', 1);
insert into payroll_management.Event_to_Payroll_Item (MapId, ItemId, EventId, IsActive) values (2, 'ILL001', 'AB02', 1);
insert into payroll_management.Event_to_Payroll_Item (MapId, ItemId, EventId, IsActive) values (3, 'BRL001', 'BR01', 1);
insert into payroll_management.Event_to_Payroll_Item (MapId, ItemId, EventId, IsActive) values (4, 'BRL001', 'BR02', 1);
insert into payroll_management.Event_to_Payroll_Item (MapId, ItemId, EventId, IsActive) values (5, 'FEL001', 'FM01', 1);
insert into payroll_management.Event_to_Payroll_Item (MapId, ItemId, EventId, IsActive) values (6, 'FEL001', 'FM02', 1);
insert into payroll_management.Event_to_Payroll_Item (MapId, ItemId, EventId, IsActive) values (7, 'LHT001', 'LT02', 1);
insert into payroll_management.Event_to_Payroll_Item (MapId, ItemId, EventId, IsActive) values (8, 'LHT002', 'LT01', 1);
insert into payroll_management.Event_to_Payroll_Item (MapId, ItemId, EventId, IsActive) values (9, 'HOT001', 'OH01', 1);
insert into payroll_management.Event_to_Payroll_Item (MapId, ItemId, EventId, IsActive) values (10, 'ROT001', 'OR02', 1);
insert into payroll_management.Event_to_Payroll_Item (MapId, ItemId, EventId, IsActive) values (11, 'ROT002', 'OR01', 1);
insert into payroll_management.Event_to_Payroll_Item (MapId, ItemId, EventId, IsActive) values (12, 'WOT001', 'OW02', 1);
insert into payroll_management.Event_to_Payroll_Item (MapId, ItemId, EventId, IsActive) values (13, 'WOT002', 'OW01', 1);